# SIDERIUS Quickstart: from a task package to a prepared run

This notebook walks through one small SIDERIUS experiment from start to finish.
The task is deliberately simple: classify 256 synthetic rows of four numbers
into two classes. It exists to show how the pieces fit together, not to train
a good model.

You will:

1. generate the example data;
2. look inside the **task package**, the folder that tells SIDERIUS what your
   data means, how it is split, what a model reads and produces, and how a
   model is scored;
3. see what SIDERIUS reads from that package, and score a set of predictions;
4. read a **workflow declaration** (`workflow.py`), the file that says which
   research steps run;
5. choose an **LLM profile** (`llm.json`), the file that says which model
   serves each step;
6. write a **preparation request** for one run and check it offline.

Sections 1 to 12 run on a CPU, offline, without API keys. Sections 13 to 15 show
the commands for preparing, launching and reading a real run. Those need
bubblewrap (the Linux sandbox SIDERIUS runs in), a provider API key and an
NVIDIA GPU, so they are shown here rather than run. Example outputs in those
sections are marked as examples; yours will differ.

**Before you start.** Install this checkout with `uv sync --group dev --frozen`.
Jupyter is not part of that environment: open this notebook with a Jupyter front
end whose kernel is this checkout's `.venv/bin/python`, started in
`examples/quickstart/`. Alternatively, paste the code cells in order into a
`.venv/bin/python` session started in that folder. The example writes its files
under `~/siderius_quickstart_workspace`; set the environment variable
`SIDERIUS_QUICKSTART_WORKSPACE` to use another folder.

## 1. Check that you are using this checkout

If several SIDERIUS checkouts exist on your machine, a notebook can silently
import another one. This cell stops if the framework code does not come from
the checkout this notebook belongs to.

In [ ]:
import json
import os
import sys
from pathlib import Path

PACK_DIR = Path.cwd().resolve()  # examples/quickstart
REPO_ROOT = PACK_DIR.parents[1]

import workflows.task_composition as task_composition

resolved = Path(task_composition.__file__).resolve()
assert resolved.is_relative_to(REPO_ROOT), (
    f"This kernel imports SIDERIUS from {resolved}, not from {REPO_ROOT}. "
    "Use this checkout's .venv/bin/python as the kernel."
)
print("checkout :", REPO_ROOT)
print("python   :", sys.version.split()[0])

Expected: your checkout path and a Python version of 3.12 or later.

## 2. Generate the example data

No data files are committed. The task ships a seeded generator and records the
SHA-256 of every file it should produce, so the data is identical on every
machine. The generator refuses to write data whose bytes differ from those
records.

In [ ]:
import importlib.util

WORKSPACE = Path(
    os.environ.get(
        "SIDERIUS_QUICKSTART_WORKSPACE", str(Path.home() / "siderius_quickstart_workspace")
    )
).resolve()


def load_plugin(name: str, filename: str):
    # Load a task plugin by path, the way SIDERIUS loads task plugin files.
    spec = importlib.util.spec_from_file_location(name, PACK_DIR / "plugins" / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


qs = load_plugin("quickstart_task", "_quickstart_task.py")
bundle = qs.materialize_run_bundle(PACK_DIR, WORKSPACE)
DATA_DIR = Path(bundle["data_dir"])
print("data folder:", DATA_DIR)
for f in sorted(DATA_DIR.iterdir()):
    print("  ", f.name, f.stat().st_size, "bytes")
print()
print("\n".join((DATA_DIR / qs.shard_filename(0)).read_text().splitlines()[:3]))

Expected: four files, `shard_0000.csv` to `shard_0003.csv` (64 rows each), and
the first rows of the first file: a sample ID, four features `x0` to `x3` and a
`label` of 0 or 1. Running the cell again rewrites the same bytes.

## 3. What is in the task package

A **task package** is a manifest plus the files it names. The manifest is the
single list SIDERIUS reads; every other file is reached through it.

| File | What it declares |
|---|---|
| `configs/task_composition/quickstart.yaml` | The manifest: which file provides each part below |
| `declared/dataset_profile.json` | How many data partitions exist and how the task labels them |
| `declared/metric_accuracy.json` | The main score: its name, whether higher or lower is better, and how it is aggregated |
| `declared/task_config.yaml` | The task description and the model contract: input and output shapes and types |
| `declared/data_manifest.json` | The SHA-256 of each generated data file |
| `plugins/_quickstart_task.py` | Data access: how rows are read, split and written out as predictions |
| `plugins/_quickstart_metrics.py` | The code that computes the main score |
| `plugins/quickstart_reference_mlp.py` | A small reference model in the shape generated models must have |

The manifest lists each part, called a *family*:

In [ ]:
import yaml

MANIFEST = REPO_ROOT / "configs" / "task_composition" / "quickstart.yaml"
manifest = yaml.safe_load(MANIFEST.read_text())
for family, value in manifest.items():
    print(f"{family:22s} {json.dumps(value)}")

Expected: one line per family, from `task_data_path` to `model_plugins`. Note
`task_health: {"none": true}`: this example declares **no Health checks**. A
Health check is a test that a trained model's output is scientifically
plausible, for example that it does not predict the same class for every row.
A real task declares its checks here and gives each one a role: **Blocking**
(a failing model is invalid and its score does not count) or **Recording** (the
result is only stored). Because this example has none, nothing in it can
invalidate a model on Health grounds; see [Health gates](../../docs/concepts/health-gates.md).

## 4. How SIDERIUS reads the task

Before any run, SIDERIUS *composes* the manifest: it loads every family, checks
that they agree, and refuses by name if anything is missing or inconsistent. It
then computes a fingerprint over the declarations and plugin code. A run records
that fingerprint, so a later resume refuses to continue if the task changed.

In [ ]:
from workflows.task_composition import compose_run_task_bindings

composition = compose_run_task_bindings(str(MANIFEST))
print("data access    :", composition.task_data_path_id)
print("main score     :", composition.metric.spec.id, f"({composition.metric.spec.direction} is better)")
print("partitions     :", composition.dataset_profile.partition_count)
print("Health checks  :", composition.task_health_binding)
print("models allowed :", tuple(composition.model_plugins.required_model_types))
print("fingerprint    :", composition.semantic_fingerprint[:16], "...")

Expected (the fingerprint will match the code in your checkout):

```text
data access    : quickstart_tabular
main score     : accuracy (higher is better)
partitions     : 4
Health checks  : explicit_none
models allowed : ('quickstart_reference_mlp',)
```

Lines starting with `[PluginLoader]` may appear first; they report model
plugins found in the checkout and are harmless.

## 5. Splits: which data trains and which evaluates

The task, not the framework, decides the split. Here shards 0 and 1 train,
shard 2 evaluates, and shard 3 is held back as a final test. The data access
plugin refuses a request that would put evaluation or test rows into training.

In [ ]:
from execute_tools.task_data_path import ScopeBuildRequest

data_path = qs.QuickstartTaskDataPath()
request = ScopeBuildRequest(round_kind="formal", selection_strategy="snapshot", portion=1.0, seed=7)
train_scope = data_path.build_training_scope(request)
eval_scope = data_path.build_eval_scope(request)
print("training rows  :", len(train_scope.rows), "from shards", sorted({r.shard for r in train_scope.rows}))
print("evaluation rows:", len(eval_scope.rows), "from shard", sorted({r.shard for r in eval_scope.rows}))
try:
    data_path.build_training_scope(
        ScopeBuildRequest(round_kind="formal", selection_strategy="target", portion=1.0, target_partitions=(3,))
    )
except ValueError as refusal:
    print("refused        :", refusal)

Expected: 128 training rows from shards `[0, 1]`, 64 evaluation rows from shard
`[2]`, and a refusal naming shard 3.

## 6. The main score

The main score is a declaration plus code. The declaration fixes the name and
the direction (higher or lower is better); SIDERIUS never guesses the direction
from the name. Before computing a number, the score checks that a prediction
file exists. A missing file gives a *not scoreable* result with a reason, never
a made-up number.

In [ ]:
from execute_tools.evaluation_metric import metric_spec_from_declaration

spec = metric_spec_from_declaration(json.loads((PACK_DIR / "declared" / "metric_accuracy.json").read_text()))
qm = load_plugin("quickstart_metrics", "_quickstart_metrics.py")
metric = qm.QuickstartAccuracyMetric(spec)
print("score:", spec.id, "| direction:", spec.direction)

probe = WORKSPACE / "metric_probe.json"
truth = {r.sample_id: r.label for r in eval_scope.rows}
probe.write_text(json.dumps(truth))
perfect = metric.evaluate({0: str(probe)}, evaluation_payload=truth, task_scope=eval_scope, data_dir=None)
wrong = metric.evaluate(
    {0: str(probe)}, evaluation_payload={k: 1 - v for k, v in truth.items()}, task_scope=eval_scope, data_dir=None
)
missing = metric.evaluate({0: str(probe) + ".gone"}, evaluation_payload=truth, task_scope=eval_scope, data_dir=None)
print("all correct :", type(perfect).__name__, perfect.scalar)
print("all wrong   :", type(wrong).__name__, wrong.scalar)
print("no file     :", type(missing).__name__, "|", missing.verdict.failures[0].detail)

Expected: `accuracy`, direction `higher`; `MetricResult 1.0` for correct
predictions, `MetricResult 0.0` for wrong ones, and `NotScoreableResult` with
"deliverable not found" when the file is missing.

## 7. The model contract

During a run, SIDERIUS's implementation step writes new model code, and its
validation step checks that the code fits the task's model contract. A model
plugin is a Python file that declares its type name, a configuration class and
the model class. The example ships one such model so you can see the shape.

In [ ]:
import torch

from execute_tools.task_data_path import EpochSamplingParams

mlp = load_plugin("quickstart_mlp", "quickstart_reference_mlp.py")
model = mlp.PLUGIN_MODEL_CLASS(mlp.PLUGIN_CONFIG_CLASS())
print("model type :", mlp.PLUGIN_MODEL_TYPE)
print("parameters :", sum(p.numel() for p in model.parameters()))

dataset = data_path.training_dataset(train_scope, EpochSamplingParams(data_dir=str(DATA_DIR), epoch_seed=0))
batch = torch.stack([dataset[i][0] for i in range(8)])
print("forward    :", tuple(batch.shape), "->", tuple(model(batch).shape))

Expected: `quickstart_reference_mlp` with 114 parameters, mapping a batch of
shape `(8, 4)` (eight rows of four features) to `(8, 2)` (two class scores).

## 8. Score a set of predictions the way a run does

In a run, a trained model writes its predictions as a file, and the score is
computed from that file. This cell does the same with stand-in predictions
(the true labels), using the composed task from section 4.

In [ ]:
from execute_tools.task_data_path import DeliverableWriteRequest, EvaluationReadRequest
from workflows.task_composition import bind_run_task_composition

identity = dict(exp_id="exp0", run_name="first_run", model_type="quickstart_reference_mlp")
prediction_dir = WORKSPACE / "predictions"

with bind_run_task_composition(composition, physical_data_root=str(DATA_DIR)):
    bound = composition.task_data_path
    predictions = {r.sample_id: r.label for r in eval_scope.rows}
    bound.write_deliverable(predictions, DeliverableWriteRequest(output_dir=str(prediction_dir), **identity))
    payload = bound.read_evaluation_payload(EvaluationReadRequest(deliverable_dir=str(prediction_dir), **identity))
    artifact = prediction_dir / bound.deliverable_name(**identity)
    outcome = composition.metric.evaluate({0: str(artifact)}, evaluation_payload=payload, task_scope=eval_scope, data_dir=None)

print("prediction file:", artifact.name)
print("score          :", type(outcome).__name__, outcome.scalar)

Expected: a file named `quickstart_predictions_quickstart_reference_mlp_first_run_exp0_0000.json`
and `MetricResult 1.0`.

## 9. The workflow declaration

The task package defines the science; a **workflow declaration** defines the
experiment. It is a short Python file, `workflow.py`, that lists the research
steps, connects them, and sets advice and training budgets. SIDERIUS reads it
without running it as a program, and reports any mistake with its line and
field before anything runs.

This cell reads the shipped starting point `reusable_training.py`, which runs
Propose → Implement → Validate → Train & tune once.
[How a workflow.py is built](../workflow_declarations/README.md#how-a-workflowpy-is-built)
explains every part, including research loops and stop conditions.

In [ ]:
from workflows.definition import parse_workflow_source

EXAMPLE_WORKFLOW = REPO_ROOT / "examples" / "workflow_declarations" / "reusable_training.py"
document = parse_workflow_source(EXAMPLE_WORKFLOW.read_text())
print("task      :", document.definition.task.manifest, "| data:", document.definition.task.data_root)
print("steps     :")
for node in document.definition.nodes:
    source = getattr(node.request, "protocol", None) or "outside input " + node.request.binding
    print(f"  {node.id:11s} {node.capability:15s} <- {source}")
print("loop      :", document.definition.loop)
print("advice    :", document.preparation_options.proposal_advice)
print("training  :", document.training_settings["launch"])

Expected:

```text
task      : task.yaml | data: data
steps     :
  hypothesis  proposal        <- outside input initial
  build       implementation  <- proposal-to-implementation
  check       validation      <- implementation-to-validation
  fit         tuning          <- validated-candidate-to-tuning
loop      : None
advice    : Prefer a small interpretable baseline
training  : {'trial_time_budget_minutes': 2.0, 'formal_time_budget_minutes': 5.0, 'healthgate_mode': 'observe_only', 'result_authority': 'diagnostic', 'is_trial': True, 'formal_portion': 1.0, 'max_rounds': 2}
```

Each step runs one research capability. The arrows name the typed connection
that carries the previous step's result into it. `loop: None` means the steps
run once. The training settings use `observe_only` and `diagnostic`: Health is
recorded but not enforced, and the scores are for exploration only. For results
you intend to compare, a workflow declares `blocking` and `scientific`.

This is the **workflow** route: you fix the path in advance. The alternative,
*orchestration*, lets your coding agent choose each research step at run time;
see [workflow or orchestration](../../README.md#choose-the-execution-workflow-or-orchestration).

## 10. Choose an LLM profile

Every run names an LLM profile in its preparation request (`llm_config_file`);
there is no implicit profile on this route. For a first flow check, copy
`configs/llm/default.json`, which uses GPT-5.6 Luna for every role. For
research runs, the [configuration guide](../../configs/llm/README.md)
recommends `openai-moderate-v1.json`. This cell lists the models each shipped
profile uses.

In [ ]:
def model_ids(node, found):
    if isinstance(node, dict):
        if "model_id" in node:
            found.add(node["model_id"])
        for value in node.values():
            model_ids(value, found)
    return found


for profile in sorted((REPO_ROOT / "configs" / "llm").glob("*.json")):
    print(f"{profile.name:26s} {sorted(model_ids(json.loads(profile.read_text()), set()))}")

Expected: one line per profile, including `default.json ['gpt-5.6-luna']` and
`openai-moderate-v1.json ['gpt-5.6-luna', 'gpt-6-astra']`.

## 11. Set up a project folder

Your own experiments live in a project folder outside the SIDERIUS checkout.
This cell turns the workspace from section 2 into one:

- `task.yaml`: a link to the shipped quickstart manifest;
- `data/`: the data from section 2;
- `workflow.py`: a copy of `reusable_training.py`;
- `llm.json`: a copy of `configs/llm/default.json`;
- `runs/`: an empty folder that will receive one subfolder per prepared run.

Existing files are left unchanged.

In [ ]:
import shutil

PROJECT = WORKSPACE
RUNS = PROJECT / "runs"
RUNS.mkdir(exist_ok=True)
if not (PROJECT / "task.yaml").exists():
    (PROJECT / "task.yaml").symlink_to(MANIFEST)
for source, target in (
    (EXAMPLE_WORKFLOW, PROJECT / "workflow.py"),
    (REPO_ROOT / "configs" / "llm" / "default.json", PROJECT / "llm.json"),
):
    if not target.exists():
        shutil.copyfile(source, target)
for path in sorted(PROJECT.iterdir()):
    print(" ", path.name + ("/" if path.is_dir() else ""))

Expected: `data/`, `llm.json`, `runs/`, `task.yaml` and `workflow.py`, plus the
`metric_probe.json` and `predictions/` written by earlier cells.

## 12. Write the preparation request and check it offline

A **preparation request** holds the settings of one run: which `workflow.py` and
`llm.json` to use, where runs go, time limits, the folders the run may read,
network access, GPU devices and the *names* of the provider key variables
(never their values). A fresh `run_id` names the run's folder.

The check below (`preflight_reusable_candidate`) reads the request and every
file it names, resolves where everything will go, and reports problems. It
writes nothing and calls no model. GPU devices are taken from this machine; on a
machine without an NVIDIA GPU the list is empty, which is accepted here, but a
launched run that trains would be refused there.

In [ ]:
import hashlib
from uuid import uuid4

from tools.candidate_prepare import ReusableCandidatePreparationRequest, preflight_reusable_candidate

workflow_file = PROJECT / "workflow.py"
gpus = tuple(sorted(str(p) for p in Path("/dev").glob("nvidia[0-9]*")))
prep_request = ReusableCandidatePreparationRequest(
    source_path=str(workflow_file),
    project_root=str(PROJECT),
    expected_source_sha256=hashlib.sha256(workflow_file.read_bytes()).hexdigest(),
    runs_root=str(RUNS),
    run_id=uuid4(),
    run_name="first_run",
    llm_config_file=str(PROJECT / "llm.json"),
    preparation_timeout_seconds=600.0,
    candidate_timeout_seconds=3600.0,
    task_read_only=(str(REPO_ROOT),),  # task.yaml links into this checkout
    candidate_network=True,
    candidate_devices=gpus,
    candidate_environment_names=("OPENAI_API_KEY",),
)
(PROJECT / "reusable-request.json").write_text(prep_request.model_dump_json(indent=2) + "\n")

plan = preflight_reusable_candidate(prep_request)
binding = plan.binding
print("request file  :", PROJECT / "reusable-request.json")
print("run folder    :", binding.envelope)
print("run workspace :", binding.locations.workspace)
print("task manifest :", binding.locations.manifest)
print("LLM profile   :", binding.request.llm_config_file)
print("GPU devices   :", list(binding.request.candidate_devices) or "none on this machine")
print("advice        :", binding.request.proposal_advice)
print("written so far:", sorted(p.name for p in RUNS.iterdir()) or "nothing")

Expected: the run folder `runs/<run_id>`, its `workspace/` subfolder, the
manifest resolved to this checkout's `quickstart.yaml`, your `llm.json`, the
GPU devices found, the advice from `workflow.py`, and `written so far: nothing`.
If a setting is wrong, for example a GPU path this machine does not have, the
cell stops with a message naming it.

Everything above ran offline. The remaining sections need more than a notebook
kernel provides, so they show the commands and example outputs.

## 13. Prepare the run

Preparation captures exactly what will run: a snapshot of `workflow.py`, the
LLM profile and every generated input, in the run folder. It runs in the
bubblewrap sandbox but calls no model and trains nothing, so it needs no API
key or GPU. From the checkout root, with your workspace path in place of
`$HOME/siderius_quickstart_workspace`:

```bash
.venv/bin/python -m tools.candidate_prepare \
  --reusable-request "$HOME/siderius_quickstart_workspace/reusable-request.json"
```

Example output (shortened; your IDs and paths will differ):

```text
{
  "run_id": "3f2b6c1e-8d4a-4e2f-9b7c-5a1d0e6f4c21",
  "envelope": "/home/you/siderius_quickstart_workspace/runs/3f2b6c1e-8d4a-4e2f-9b7c-5a1d0e6f4c21",
  "status": "prepared",
  ...
}
```

Success is exit status 0 with `"status": "prepared"`. The `envelope` is the run
folder; `workflow.py.snapshot` there is the exact workflow that will run. A
request can be prepared once: to prepare again, rerun section 12 for a fresh
`run_id`.

## 14. Launch and follow the run

Launching calls paid APIs and trains on the GPU. Save `store.json` in your
project, telling the operations tool where prepared runs live and where to keep
its own records. For `state_root`, name a separate folder that does not exist
yet (its parent must). The first submit creates it with private permissions,
and later submits reuse it. Do not create it yourself.

```json
{
  "state_root": "/home/you/siderius_quickstart_workspace/operations",
  "preparation_roots": ["/home/you/siderius_quickstart_workspace/runs"],
  "candidate_roots": ["/home/you/siderius_quickstart_workspace/runs"]
}
```

Turn the prepared run folder into a launch request, then read it before you
submit:

```python
from pathlib import Path
from uuid import uuid4

from tools.candidate_operations import OperationStoreConfig, build_reusable_operation
from tools.candidate_prepare import select_reusable_preparation

project = Path.home() / "siderius_quickstart_workspace"
config = OperationStoreConfig.model_validate_json((project / "store.json").read_bytes())
selection = select_reusable_preparation(str(project / "runs" / "<run_id>"))
submission = build_reusable_operation(config, selection, uuid4())
(project / "reusable-submission.json").write_text(submission.model_dump_json(indent=2))
```

Make `OPENAI_API_KEY` available in your shell (see
[API keys](../../docs/getting-started/installation.md#api-keys)), then submit
and follow the run:

```bash
.venv/bin/python -m tools.candidate_operations submit \
  --store-config "$HOME/siderius_quickstart_workspace/store.json" \
  --request "$HOME/siderius_quickstart_workspace/reusable-submission.json"
.venv/bin/python -m tools.candidate_operations status \
  --store-config "$HOME/siderius_quickstart_workspace/store.json" --operation-id OPERATION_UUID
.venv/bin/python -m tools.candidate_operations log \
  --store-config "$HOME/siderius_quickstart_workspace/store.json" --operation-id OPERATION_UUID \
  --kind native --offset 0 --limit 65536
```

`submit` prints an `operation_id`; exit status 0 means the run was admitted,
not that it succeeded. `status` prints JSON whose `state` moves from `active`
to `completed`. Example (shortened):

```text
{
  "operation_id": "8c1d...",
  "state": "completed",
  "receipt": {"status": "completed", ...},
  ...
}
```

## 15. Read the result

A finished run is a record, not a verdict. Each training step saves its report
as `run_output_<run_name>.json` inside the run folder. Its most important fields:

- `best_valid_formal_denoising_score`: the best main-score value among Formal
  results that passed Health. It is the number to compare between runs. (The
  field name is historical; it holds this task's accuracy.)
- `all_records`: every training attempt, with its status, score and each Health
  check's outcome;
- `termination_reason`: why training stopped.

A result counts scientifically only when the workflow declared `blocking` and
`scientific`, the score could be computed, and every Blocking Health check
passed. This example declares `diagnostic`, so its scores are for exploration
only. [Read the result](../../docs/getting-started/README.md#6-read-the-result)
explains every layer, and the browser console's **Monitor** page shows the same
report. This cell prints the key fields of any finished run in your workspace:

In [ ]:
reports = sorted(RUNS.glob("*/workspace/**/run_output_*.json"))
if not reports:
    print("No finished run yet: prepare and launch one (sections 13 and 14), then rerun this cell.")
for report_path in reports:
    report = json.loads(report_path.read_text())
    print(report_path.relative_to(RUNS))
    for field in ("status", "termination_reason", "best_valid_formal_denoising_score", "result_authority"):
        print(f"  {field:35s} {report.get(field)}")

Offline, the cell prints `No finished run yet`.

## 16. Use your own task

To run SIDERIUS on your own data, open this checkout in your coding agent and
follow [getting started](../../docs/getting-started/README.md). For a
classification dataset in CSV files, start from the
[CSV classification template](../task_templates/csv_classification/README.md);
for any other task, see the [task definition guide](../../docs/guides/define-a-task.md).
The same steps apply: a task package, a `workflow.py`, an `llm.json` and a
preparation request, then prepare, launch and read the result.